# Nemotron 3 Nano - Advanced SFT: Maximize Min-Log-Prob (worst-token / branch-weighted)

Same scaffold as `nvidia-nemotron-sft-standard.ipynb`. Swaps the standard mean-NLL loss for **selectable advanced objectives** that target hard, low-confidence tokens (the dominant failure mode on logical-puzzle reasoning).

## Loss menu
Switch via `LOSS_MODE`:

| mode | formula | notes |
|---|---|---|
| `mean`      | `mean(NLL)` over assistant tokens | baseline / warm-up |
| `minmax`    | `mean_batch( max_seq(NLL) )` = `mean_batch( -min_seq(log p) )` | maximize worst-token logp per seq; sparse |
| `topk_min`  | `mean_batch( mean( top-K NLL ) )` | smoothed worst-K, less sparse than `minmax` |
| `blend`     | `alpha*mean + (1-alpha)*topk_min` | proven SFT base + worst-token push |
| `branch`    | NLL re-weighted by `min(1, |log p|/branch_logprob)` | Tong Huikang Progress Prize trick; upweights confident-wrong tokens |

## Research provenance
Loss design drawn from:
- **Tong Huikang Progress Prize** (`tonghuikang/nemotron`) — `loss_config.py` `CrossEntropyWithWeightingLossConfig`: branch weight `min(1, |logp|/branch_logprob)` upweights tokens where the model is decisively wrong.
- **Critical Token Fine-Tuning** (Wang 2025, arXiv:2510.10974) — train on counterfactually-critical token subset only; CFT consistently outperforms standard SFT on math reasoning benchmarks (Qwen / OLMo / LLaMA).
- **Beyond Log Likelihood** (Wu 2025, arXiv:2510.00526) — probability-based objectives across model capability continuum; min-probability is a special case.
- **SFT-GO / SFTKey** (arXiv:2512.21017) — worst-group + key-token re-weighting, +5pp avg over uniform NLL.

## Compute-safety
Same as standard notebook: `SMOKE_TEST` first, subset+1-2 epochs, `TRAIN_MAX_LEN=4096`, memory-safe fused CE, 3-tier save+recovery, `/kaggle/working/submission.zip`, postprocess integration.

In [1]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0}


In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

Found Triton wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
triton spec: ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7b8590cfcda0>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

Training environment fixes applied.


In [4]:
import os

BASE_MODEL_NAME   = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SFT_DATA_PATH     = "/kaggle/input/datasets/ramkan07/v7-mix/new_dataset_filtered2.csv"

OUTPUT_ROOT       = "outputs"
SFT_ADAPTER_DIR   = os.path.join(OUTPUT_ROOT, "minmax_logprob_adapter")
SUBMISSION_DIR    = os.path.join(OUTPUT_ROOT, "submission_minmax_logprob")
TB_LOG_DIR        = os.path.join(OUTPUT_ROOT, "tb_logs_minmax_logprob")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED = 42

# ===========================================================================
# COMPUTE-SAFETY KNOBS
# ===========================================================================
SMOKE_TEST  = 0
SMOKE_ROWS  = 64
SMOKE_STEPS = 8
SUBSET_N    = 3000
NUM_EPOCHS  = 2
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 8192
STRATIFIED_BATCHING = True

# ===========================================================================
# LOSS KNOBS
# ===========================================================================
# mean      = safe baseline (standard NLL)
# minmax    = pure worst-token (sparse, slow; use after warm-up)
# topk_min  = smoothed worst-K (recommended advanced default)
# blend     = alpha*mean + (1-alpha)*topk_min (best of both worlds)
# branch    = NLL re-weighted by min(1, |logp|/branch_logprob)
LOSS_MODE = "blend"

# topk_min / blend: number of WORST tokens per sequence to average
TOPK_MIN  = 16

# blend: convex combination weight for mean component
BLEND_ALPHA = 0.5

# branch: scale at which weight saturates (smaller -> harder upweighting)
BRANCH_LOGPROB = 1.0

# Optional warm-up: train with mean-NLL for the first N steps before switching
# to LOSS_MODE. Stabilizes early training when worst tokens are huge & noisy.
WARMUP_MEAN_STEPS = 0

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print({"SMOKE_TEST": SMOKE_TEST, "SUBSET_N": SUBSET_N, "NUM_EPOCHS": NUM_EPOCHS,
       "TRAIN_MAX_LEN": TRAIN_MAX_LEN, "LOSS_MODE": LOSS_MODE, "TOPK_MIN": TOPK_MIN,
       "BLEND_ALPHA": BLEND_ALPHA, "BRANCH_LOGPROB": BRANCH_LOGPROB,
       "WARMUP_MEAN_STEPS": WARMUP_MEAN_STEPS})

{'SMOKE_TEST': 0, 'SUBSET_N': 3000, 'NUM_EPOCHS': 2, 'TRAIN_MAX_LEN': 8192, 'LOSS_MODE': 'blend', 'TOPK_MIN': 16, 'BLEND_ALPHA': 0.5, 'BRANCH_LOGPROB': 1.0, 'WARMUP_MEAN_STEPS': 0}


In [5]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )
    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

Torch: 2.10.0+cu128 CUDA: True


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Offline package installation finished.


In [6]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MODEL_MAX_LEN,
        load_in_4bit=False, load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-31 16:04:21.567436: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780243461.728309     164 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780243461.781363     164 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780243462.208636     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780243462.208651     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780243462.208652     164 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded with Unsloth.


## LoRA Targets (RSLoRA + DoRA, r=32, sensitive-module priority)

In [7]:
from peft import LoraConfig, get_peft_model, TaskType
import re

linear_modules = []
for name, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

target_regex = (
    r".*("
    r"self_attn\.(q|k|v|o)_proj"
    r"|mamba\.(in|out|x|dt|gate)_proj"
    r"|shared_experts\.(gate|up|down)_proj"
    r")$"
)
matched = [n for n in linear_modules if re.match(target_regex, n)]
print(f"LoRA target regex matched {len(matched)} modules.")
if len(matched) == 0:
    sample = [n for n in linear_modules if "expert" in n or "mamba" in n or "self_attn" in n][:20]
    raise RuntimeError(f"LoRA target_regex matched 0 modules. Sample: {sample}")

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
    target_modules=target_regex,
    task_type=TaskType.CAUSAL_LM,
    use_rslora=True,
    use_dora=True,
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(s for _, s in trainable)
# print(f"[audit] {len(trainable)} trainable tensors  total={n_train/1e6:.1f}M")
# print(f"[audit] first 5: {[n for n,_ in trainable[:5]]}")
# print(f"[audit] last  5: {[n for n,_ in trainable[-5:]]}")
# assert 50_000_000 <= n_train <= 400_000_000, \
#     f"[audit] trainable count {n_train/1e6:.1f}M outside [50M, 400M] -- inspect target_regex."

LoRA target regex matched 46 modules.
trainable params: 9,568,000 || all params: 31,587,505,344 || trainable%: 0.0303


## Generalized System Prompt (identical at train + eval)

In [8]:
SYSTEM_PROMPT = """You are a meticulous reasoning engine for a logical-puzzle benchmark. \
Every task has exactly one correct, deterministic answer that can be derived by \
exact rule-following and arithmetic. Speed does not matter; correctness does.

GENERAL METHOD
1. Read the problem twice. Identify the category and restate, in your own words, \
the exact transformation or quantity being asked for.
2. Extract every given value, rule, mapping, base, unit, and constant verbatim. \
Never invent data that is not stated.
3. Work strictly step by step. Show each intermediate result. Do arithmetic \
digit by digit and re-check it. When a rule is defined in the prompt, apply it \
literally rather than relying on prior assumptions.
4. Verify: substitute your answer back into the problem and confirm it satisfies \
every stated condition. If it does not, find your error and redo the step.

CATEGORY-SPECIFIC RULES
- Bit manipulation: operate on the stated bit width (default 8 bits). Preserve \
leading zeros. Apply AND/OR/XOR/NOT/shifts exactly; for shifts state whether bits \
fall off or wrap as specified. Report the result in the format the prompt uses.
- Number-base conversion: convert through base 10 as an intermediate when helpful. \
Map digits A-F carefully. State the source and target base. No prefixes unless asked.
- Gravitational constant / free-fall: use the constant exactly as given in the \
prompt (do not substitute a textbook g). Track units. Round only at the end.
- Unit conversion: write the conversion factor as an explicit fraction, cancel \
units, keep full precision until the final rounding. State the final unit.
- Text encryption / cipher: determine the exact scheme and direction. Transform \
one character at a time, preserving case, spacing, and punctuation unless told otherwise.
- Algebraic equations & equation transformation: isolate the target symbol with \
inverse operations; or apply the defined transformation rule literally. Keep equations balanced.

OUTPUT CONTRACT (mandatory)
- First think inside a single <think> ... </think> block containing your full \
step-by-step derivation and verification.
- Immediately after </think>, output the final answer once, wrapped exactly as \
\\boxed{...} with nothing after it.
- The boxed content must be only the answer in the form the problem expects \
(e.g. an 8-bit binary string, a number, a word, or an expression) - no units \
unless the problem asks for them, no extra words."""

print(f"SYSTEM_PROMPT chars: {len(SYSTEM_PROMPT)}")

SYSTEM_PROMPT chars: 2417


## Dataset Prep + Assistant-Only Masking

In [9]:
import pandas as pd, re
from datasets import Dataset as HFDataset

df_sft = pd.read_csv(SFT_DATA_PATH)
print(f"SFT data: {len(df_sft)} rows.  Columns: {list(df_sft.columns)}")

def _find(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n in low: return low[n]
    return None

PROMPT_COL = _find(df_sft.columns, ["prompt", "question", "problem", "input"])
ANSWER_COL = _find(df_sft.columns, ["answer", "solution", "label", "target", "final_answer"])
COT_COL    = _find(df_sft.columns, ["cot", "reasoning", "think", "generated_cot",
                                    "response", "completion", "rationale", "output"])
TYPE_COL   = _find(df_sft.columns, ["type", "category", "puzzle_type", "task_type"])
print(f"Detected -> prompt={PROMPT_COL!r}  answer={ANSWER_COL!r}  cot={COT_COL!r}  type={TYPE_COL!r}")
if PROMPT_COL is None:
    raise ValueError(f"No prompt-like column in {list(df_sft.columns)}")

df_sft = df_sft.dropna(subset=[PROMPT_COL]).reset_index(drop=True)
df_sft = df_sft.sample(frac=1, random_state=SEED).reset_index(drop=True)

if SMOKE_TEST:
    df_sft = df_sft.head(SMOKE_ROWS).reset_index(drop=True)
    print(f"[SMOKE] using {len(df_sft)} rows")
elif SUBSET_N is not None:
    df_sft = df_sft.head(SUBSET_N).reset_index(drop=True)
    print(f"[REAL] using subset of {len(df_sft)} rows")
else:
    print(f"[REAL] using all {len(df_sft)} rows")

def build_assistant_text(row):
    """Always emit the canonical contract: <think>\n{reasoning}\n</think>\n\\boxed{ans}.

    CRITICAL FIX (zero-loss bug): the old version, when the cot already contained
    \\boxed{}, returned the raw cot with `<think>` prepended but NO closing
    `</think>` and the boxed answer buried inside the think block. Under
    `enable_thinking=True` the Nemotron/Qwen3 chat template mishandles an assistant
    turn that opens <think> but never closes it -> the assistant content gets
    stripped at render time -> full_text == prefix_text -> the prompt-prefix mask
    masks EVERY token -> every row is dropped by the unmasked-token filter ->
    tokenized_ds is empty -> trainer logs loss = 0.0 forever.

    Here we instead strip the cot's own boilerplate/box and rebuild a well-formed
    target with a real </think> and exactly one trailing \\boxed{answer} taken from
    the answer column (the canonical ground truth).
    """
    ans = "" if ANSWER_COL is None else str(row[ANSWER_COL]).strip()
    cot = "" if COT_COL is None else str(row.get(COT_COL, "") or "").strip()

    # drop any existing think wrapper so we control the tags
    cot = cot.replace("<think>", "").replace("</think>", "").strip()
    # drop boilerplate "I will put/return my final answer inside \boxed{}" lines
    cot = re.sub(r'(?im)^.*I will (now )?(put|return) .*\\boxed\{\}.*$', '', cot)
    # drop trailing "The answer (in \boxed{x}) is \boxed{y}" lines
    cot = re.sub(r'(?im)^.*The answer .*\\boxed\{[^}]*\}.*$', '', cot)
    # remove any remaining flat \boxed{...} left inside the reasoning
    cot = re.sub(r'\\boxed\{[^{}]*\}', '', cot)
    # collapse the blank lines those deletions leave behind
    cot = re.sub(r'\n{3,}', '\n\n', cot).strip()

    think = cot if cot else "Work through the problem step by step."
    return f"<think>\n{think}\n</think>\n\\boxed{{{ans}}}"

records = []
record_types = []
for _, row in df_sft.iterrows():
    records.append({
        "system":    SYSTEM_PROMPT,
        "user":      str(row[PROMPT_COL]) + PROMPT_SUFFIX,
        "assistant": build_assistant_text(row),
    })
    record_types.append(str(row[TYPE_COL]) if TYPE_COL else "unknown")

# Sanity: every target must close </think> and end with exactly one \boxed{}
_bad = [i for i, r in enumerate(records)
        if "</think>" not in r["assistant"]
        or r["assistant"].count("\\boxed{") != 1
        or not r["assistant"].rstrip().endswith("}")]
if _bad:
    raise ValueError(f"{len(_bad)} malformed assistant targets (e.g. idx {_bad[:5]}). "
                     f"First: {records[_bad[0]]['assistant'][-200:]!r}")
print(f"[format] all {len(records)} targets well-formed: <think>..</think> + single trailing \\boxed{{}}")

raw_ds = HFDataset.from_list(records)
print(f"SFT records: {len(records)}")
print("Type distribution:", dict(pd.Series(record_types).value_counts().head(10).to_dict()))
print("\n--- sample assistant target HEAD ---\n", records[0]["assistant"][:300])
print("\n--- sample assistant target TAIL ---\n", records[0]["assistant"][-160:])

SFT data: 9156 rows.  Columns: ['id', 'prompt', 'answer', 'type', 'generated_cot']
Detected -> prompt='prompt'  answer='answer'  cot='generated_cot'  type='type'
[REAL] using subset of 3000 rows
[format] all 3000 targets well-formed: <think>..</think> + single trailing \boxed{}
SFT records: 3000
Type distribution: {'gravity': 520, 'cipher': 515, 'unit_conversion': 508, 'bit_manipulation': 505, 'numeral': 496, 'equation_numeric_deduce': 190, 'cryptarithm_deduce': 182, 'cryptarithm_guess': 47, 'equation_numeric_guess': 37}

--- sample assistant target HEAD ---
 <think>
We need to find the encryption mapping from the examples. It looks like a substitution cipher.

Listing the input words:

【xgbbdc hznuljdcn bxd qvnbdczlfn qdnngod】
xgbbdc
 hznuljdcn
 bxd
 qvnbdczlfn
 qdnngod

【bxd ulylcefy izko elfkh】
bxd
 ulylcefy
 izko
 elfkh

【bxd xzhhdk xgbbdc hcdgqn】
bx

--- sample assistant target TAIL ---
 y is
rabbit chases in garden

Iterating over the unknown letters to see if they are in the que

In [10]:
def tokenize_with_assistant_mask(example):
    full_msgs = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    prefix_msgs = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]
    def render(msgs, add_gen):
        try:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen)
    full_text   = render(full_msgs,   False)
    prefix_text = render(prefix_msgs, True)
    full_ids   = tokenizer(full_text,   add_special_tokens=False, truncation=True,
                           max_length=TRAIN_MAX_LEN)["input_ids"]
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    cutoff = min(len(prefix_ids), len(full_ids))
    labels = list(full_ids)
    for i in range(cutoff):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}

tokenized_ds = raw_ds.map(
    tokenize_with_assistant_mask,
    remove_columns=raw_ds.column_names,
    desc="Tokenize + assistant mask",
)

_kept_types = []
_kept_rows  = []
_n_all_masked = 0
for i, ex in enumerate(tokenized_ds):
    if any(t != -100 for t in ex["labels"]):
        _kept_rows.append(ex)
        _kept_types.append(record_types[i])
    else:
        _n_all_masked += 1
tokenized_ds = HFDataset.from_list(_kept_rows)
record_types = _kept_types
print(f"Kept {len(tokenized_ds)} rows with >=1 unmasked assistant token "
      f"(dropped {_n_all_masked} fully-masked).")

# ---- HARD GUARD: empty dataset is the silent cause of loss == 0.0 -----------
# If the prompt-prefix mask wipes every token, all rows are dropped, the trainer
# sees an empty dataloader, and logs loss 0.0. Fail loudly here instead.
if len(tokenized_ds) == 0:
    raise RuntimeError(
        "tokenized_ds is EMPTY -> every row was fully masked. The assistant-mask "
        "prefix trick failed (full_text == prefix_text). This is the loss==0.0 bug. "
        "Check that assistant targets contain a closing </think> and that the chat "
        "template renders the assistant content after the generation prompt."
    )
if _n_all_masked > 0.10 * (len(tokenized_ds) + _n_all_masked):
    print(f"[WARN] {_n_all_masked} rows ({_n_all_masked/(len(tokenized_ds)+_n_all_masked)*100:.0f}%) "
          f"were fully masked -- masking may be partially broken.")

# ---- VISIBLE DIAGNOSTIC: decode the unmasked span of row 0 ------------------
# The unmasked region MUST be the assistant content (reasoning + boxed answer),
# NOT the prompt. If you see system/user text here, masking is misaligned.
_ex0 = tokenized_ds[0]
_unmasked_ids = [tid for tid, lab in zip(_ex0["input_ids"], _ex0["labels"]) if lab != -100]
print(f"\n[mask-check] row0: total={len(_ex0['input_ids'])} tokens, "
      f"unmasked(assistant)={len(_unmasked_ids)}")
print("[mask-check] decoded unmasked span (should be the assistant answer):")
print(repr(tokenizer.decode(_unmasked_ids)[:400]))
_dec = tokenizer.decode(_unmasked_ids)
assert "\\boxed{" in _dec or "boxed" in _dec, \
    "[mask-check] unmasked span has no boxed answer -> masking is misaligned, loss will be garbage."

import numpy as np
_lens = np.array([len(x["input_ids"]) for x in tokenized_ds])
_ulens = np.array([sum(1 for l in x["labels"] if l != -100) for x in tokenized_ds])
pct = lambda p: int(np.percentile(_lens, p))
print(f"\nToken length  min={_lens.min()}  mean={_lens.mean():.0f}  "
      f"p50={pct(50)}  p90={pct(90)}  p99={pct(99)}  max={_lens.max()}")
print(f"Unmasked/seq  min={_ulens.min()}  mean={_ulens.mean():.0f}  max={_ulens.max()}")

Tokenize + assistant mask:   0%|          | 0/3000 [00:00<?, ? examples/s]

Kept 3000 rows with >=1 unmasked assistant token (dropped 0 fully-masked).

[mask-check] row0: total=2761 tokens, unmasked(assistant)=2085
[mask-check] decoded unmasked span (should be the assistant answer):
'We need to find the encryption mapping from the examples. It looks like a substitution cipher.\n\nListing the input words:\n\n【xgbbdc hznuljdcn bxd qvnbdczlfn qdnngod】\nxgbbdc\n hznuljdcn\n bxd\n qvnbdczlfn\n qdnngod\n\n【bxd ulylcefy izko elfkh】\nbxd\n ulylcefy\n izko\n elfkh\n\n【bxd xzhhdk xgbbdc hcdgqn】\nbxd\n xzhhdk\n xgbbdc\n hcdgqn\n\n【xgbbdc hcdgqn tlli】\nxgbbdc\n hcdgqn\n tlli\n\n【 cgttzb uxgndn zk ogchdk】\n cgttzb\n ux'

Token length  min=644  mean=3821  p50=3554  p90=7391  p99=8046  max=8192
Unmasked/seq  min=43  mean=3155  max=7421


In [11]:
import torch

class CompletionOnlyDataCollator:
    """Pads input_ids/labels/attention_mask; preserves -100 on prompt tokens."""
    def __init__(self, tokenizer, label_pad_id=-100):
        self.pad_id = tokenizer.pad_token_id
        self.label_pad_id = label_pad_id
    def __call__(self, features):
        maxlen = max(len(f["input_ids"]) for f in features)
        input_ids, labels, attn = [], [], []
        for f in features:
            ids = list(f["input_ids"]); lab = list(f["labels"])
            pad = maxlen - len(ids)
            input_ids.append(ids + [self.pad_id] * pad)
            labels.append(lab + [self.label_pad_id] * pad)
            attn.append([1] * len(ids) + [0] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = CompletionOnlyDataCollator(tokenizer)
print("Collator ready.")

Collator ready.


## Advanced Trainer (selectable loss)

All loss modes compute per-token NLL via fused `F.cross_entropy(reduction='none')` (no fp32 full-vocab `log_softmax` spike). Then reduce per mode:

- **mean**: `nll[mask].mean()` — vanilla.
- **minmax**: `mean_batch( max_seq( nll_masked ) )` — pure worst-token. Tightens hardest position only.
- **topk_min**: `mean_batch( mean( topk(nll_masked, K) ) )` — smoothed worst-K. Less sparse gradient than `minmax`, more focus than `mean`.
- **blend**: `alpha*mean + (1-alpha)*topk_min` — RECOMMENDED. Keeps gradient density of mean while adding worst-K pressure.
- **branch**: `sum(nll * w_branch) / sum(w_branch)` with `w_branch = min(1, |logp|/branch_logprob)`. Tong Huikang trick — tokens where model is decisively wrong get full weight; near-saturated tokens get downweighted.

In [12]:
import os, sys
os.environ["TORCHDYNAMO_DISABLE"]   = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch, torch._dynamo
torch._dynamo.config.disable = True
torch._dynamo.reset()

for _m in list(sys.modules):
    if _m == "trl" or _m.startswith("trl.") or "unsloth" in _m.lower():
        del sys.modules[_m]
sys.meta_path = [f for f in sys.meta_path
                 if "unsloth" not in type(f).__module__.lower()]

import torch.nn.functional as F
from trl import SFTTrainer, SFTConfig
assert "unsloth" not in SFTTrainer.__module__.lower(), \
    f"Still using Unsloth trainer: {SFTTrainer.__module__}"
print(f"SFTTrainer module: {SFTTrainer.__module__}  (vanilla TRL, dynamo disabled)")

from torch.utils.data import DataLoader, Sampler
from collections import defaultdict
import random, math

def build_stratified_index_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1
    order = [idx for batch in batches for idx in batch]
    if len(order) != len(labels):
        raise ValueError("Stratified order size mismatch")
    return order

class PrecomputedOrderSampler(Sampler):
    def __init__(self, order):
        self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)

class AdvancedSFTTrainer(SFTTrainer):
    """SFT with selectable advanced loss (mean / minmax / topk_min / blend / branch).

    Memory-safe: fused F.cross_entropy(reduction='none') -> per-token NLL without
    materializing fp32 (seq x vocab) log_softmax tensor. Reduction varies by mode.
    `**kwargs` swallows num_items_in_batch from newer transformers.
    """
    def __init__(self, *args, loss_mode="mean", topk_min=16, blend_alpha=0.5,
                 branch_logprob=1.0, warmup_mean_steps=0, stratified_order=None,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_mode = loss_mode
        self.topk_min = topk_min
        self.blend_alpha = blend_alpha
        self.branch_logprob = branch_logprob
        self.warmup_mean_steps = warmup_mean_steps
        self.stratified_order = stratified_order

    def _current_mode(self):
        if self.state.global_step < self.warmup_mean_steps:
            return "mean"
        return self.loss_mode

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits                  # (B, T, V) bf16
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:].to(shift_logits.device)
        B, T, V = shift_logits.shape

        nll_flat = F.cross_entropy(
            shift_logits.reshape(-1, V),
            shift_labels.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, T)                              # -log p(true), 0 where ignored
        mask = (shift_labels != -100)             # (B, T) bool

        mode = self._current_mode()
        if mode == "mean":
            loss = nll_flat[mask].mean()
        elif mode == "minmax":
            nll_masked = nll_flat.masked_fill(~mask, float("-inf"))
            worst, _ = nll_masked.max(dim=-1)
            valid = mask.any(dim=-1)
            loss = worst[valid].mean()
        elif mode == "topk_min":
            nll_masked = nll_flat.masked_fill(~mask, float("-inf"))
            k = min(self.topk_min, T)
            topk_vals, _ = torch.topk(nll_masked, k=k, dim=-1)
            valid_count = mask.sum(dim=-1).clamp(min=1)
            eff_k = torch.minimum(
                torch.full_like(valid_count, k),
                valid_count,
            )
            real_topk = torch.where(
                topk_vals == float("-inf"),
                torch.zeros_like(topk_vals),
                topk_vals,
            )
            per_seq = real_topk.sum(dim=-1) / eff_k.to(real_topk.dtype)
            valid_seq = mask.any(dim=-1)
            loss = per_seq[valid_seq].mean()
        elif mode == "blend":
            mean_loss = nll_flat[mask].mean()
            nll_masked = nll_flat.masked_fill(~mask, float("-inf"))
            k = min(self.topk_min, T)
            topk_vals, _ = torch.topk(nll_masked, k=k, dim=-1)
            valid_count = mask.sum(dim=-1).clamp(min=1)
            eff_k = torch.minimum(
                torch.full_like(valid_count, k), valid_count,
            )
            real_topk = torch.where(
                topk_vals == float("-inf"),
                torch.zeros_like(topk_vals), topk_vals,
            )
            per_seq = real_topk.sum(dim=-1) / eff_k.to(real_topk.dtype)
            valid_seq = mask.any(dim=-1)
            tk_loss = per_seq[valid_seq].mean()
            loss = self.blend_alpha * mean_loss + (1.0 - self.blend_alpha) * tk_loss
        elif mode == "branch":
            # Branch weight (Tong Huikang Progress Prize loss_config.py):
            #   w = min(1, |log p| / branch_logprob)
            # log p = -nll. Tokens with very small |log p| (near-saturated, model
            # already correct OR very far from correct) get downweighted; the
            # mid-confidence range gets full weight.
            nll_masked = nll_flat * mask.to(nll_flat.dtype)
            w_branch = (nll_masked / self.branch_logprob).clamp(max=1.0)
            w_branch = w_branch * mask.to(w_branch.dtype)
            num = (nll_masked * w_branch).sum()
            den = w_branch.sum().clamp(min=1.0)
            loss = num / den
        else:
            raise ValueError(f"bad loss_mode={mode}")

        if not torch.isfinite(loss):
            loss = (logits.sum() * 0.0).requires_grad_(True)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        if len(self.stratified_order) != len(self.train_dataset):
            raise ValueError("Stratified order length mismatch")
        dataloader_kwargs = {
            "batch_size": self.args.per_device_train_batch_size,
            "sampler": PrecomputedOrderSampler(self.stratified_order),
            "collate_fn": self.data_collator,
            "num_workers": self.args.dataloader_num_workers,
            "pin_memory": self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
            "drop_last": self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dataloader_kwargs)

print("AdvancedSFTTrainer defined.")

SFTTrainer module: trl.trainer.sft_trainer  (vanilla TRL, dynamo disabled)
AdvancedSFTTrainer defined.


In [13]:
_max_steps = SMOKE_STEPS if SMOKE_TEST else -1

sft_config = SFTConfig(
    output_dir                   = os.path.join(OUTPUT_ROOT, "minmax_logprob_run"),
    num_train_epochs             = NUM_EPOCHS,
    max_steps                    = _max_steps,
    per_device_train_batch_size  = 2,
    gradient_accumulation_steps  = 4,
    learning_rate                = 8e-5,
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.05,
    weight_decay                 = 0.01,
    max_grad_norm                = 1.0,
    optim                        = "paged_adamw_8bit",
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.95,
    adam_epsilon                 = 1e-8,
    bf16                         = True,
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": True},
    max_length                   = TRAIN_MAX_LEN,
    packing                      = False,
    dataset_kwargs               = {"skip_prepare_dataset": True},
    remove_unused_columns        = False,
    logging_steps                = 1 if SMOKE_TEST else 10,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "none",
    save_strategy                = "no" if SMOKE_TEST else "steps",
    save_steps                   = 100,
    save_total_limit             = 2,
    seed                         = SEED,
    dataloader_num_workers       = 2,
)

effective_batch_size = max(1, sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
stratified_order = None
if STRATIFIED_BATCHING and len(set(record_types)) > 1:
    stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
    print(f"Stratified order built (eff batch={effective_batch_size})")
else:
    print("Stratified batching disabled or single type -> default shuffle.")

print(f"SFTConfig ready. mode={'SMOKE' if SMOKE_TEST else 'REAL'}  max_steps={_max_steps}  "
      f"epochs={NUM_EPOCHS}  eff_batch={effective_batch_size}  max_length={TRAIN_MAX_LEN}")
print(f"Loss: mode={LOSS_MODE}  topk_min={TOPK_MIN}  blend_alpha={BLEND_ALPHA}  "
      f"branch_logprob={BRANCH_LOGPROB}  warmup_mean_steps={WARMUP_MEAN_STEPS}")

Stratified order built (eff batch=8)
SFTConfig ready. mode=REAL  max_steps=-1  epochs=2  eff_batch=8  max_length=8192
Loss: mode=blend  topk_min=16  blend_alpha=0.5  branch_logprob=1.0  warmup_mean_steps=0


## Launch + Robust Save

In [14]:
import gc, time, torch, os, glob, shutil

trainer = AdvancedSFTTrainer(
    model              = model,
    args               = sft_config,
    train_dataset      = tokenized_ds,
    data_collator      = data_collator,
    processing_class   = tokenizer,
    loss_mode          = LOSS_MODE,
    topk_min           = TOPK_MIN,
    blend_alpha        = BLEND_ALPHA,
    branch_logprob     = BRANCH_LOGPROB,
    warmup_mean_steps  = WARMUP_MEAN_STEPS,
    stratified_order   = stratified_order,
)

_ckpts = sorted(glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")),
                key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = bool(_ckpts) and not SMOKE_TEST
print(f"{'Resuming from' if resume else 'Fresh start; no'} checkpoint in {sft_config.output_dir}")

torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

train_err = None
try:
    trainer.train(resume_from_checkpoint=resume)
    print(f"Training done in {(time.time()-t0)/60:.1f} min")
except Exception as e:
    train_err = e
    print(f"[TRAIN ERROR after {(time.time()-t0)/60:.1f} min] {type(e).__name__}: {e}")
    print("[recovery] will still try to save whatever progressed so far.")

print(f"PEAK VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

def _save_adapter(dest):
    os.makedirs(dest, exist_ok=True)
    needed = ["adapter_config.json", "adapter_model.safetensors"]
    try:
        inner = trainer.model
        if hasattr(inner, "save_pretrained"):
            inner.save_pretrained(dest)
        tokenizer.save_pretrained(dest)
    except Exception as e:
        print(f"[save] inner.save_pretrained failed: {e}; trying trainer.save_model")
        try:
            trainer.save_model(dest); tokenizer.save_pretrained(dest)
        except Exception as e2:
            print(f"[save] trainer.save_model also failed: {e2}")
    missing = [n for n in needed if not os.path.exists(os.path.join(dest, n))]
    if missing:
        ckpts = sorted(glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")),
                       key=lambda p: int(p.rsplit("-", 1)[-1]))
        if ckpts:
            src = ckpts[-1]
            print(f"[save] {missing} missing in {dest}; copying from {src}")
            for fname in needed:
                sp = os.path.join(src, fname)
                if os.path.exists(sp):
                    shutil.copy2(sp, os.path.join(dest, fname))
    have = {n: os.path.exists(os.path.join(dest, n)) for n in needed}
    sizes = {n: (os.path.getsize(os.path.join(dest, n)) / 1024 / 1024 if have[n] else 0)
             for n in needed}
    print(f"[save] {dest} -> have={have}  sizes_MB={ {k: f'{v:.1f}' for k, v in sizes.items()} }")
    return all(have.values())

ok = _save_adapter(SFT_ADAPTER_DIR)
if not ok:
    print("[save] WARNING: required files still missing. Checkpoint dirs:",
          glob.glob(os.path.join(sft_config.output_dir, "checkpoint-*")))
else:
    print(f"Adapter saved + verified -> {SFT_ADAPTER_DIR}")

if SMOKE_TEST:
    print("\n[SMOKE] green if no OOM/nan + PEAK VRAM has headroom. Set SMOKE_TEST=0 and rerun.")

if train_err is not None:
    raise train_err

Fresh start; no checkpoint in outputs/minmax_logprob_run
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000


KeyboardInterrupt: 

## Greedy Sanity Check

In [ ]:
import torch
model.eval()
try: model.config.use_cache = True
except Exception: pass
_probe = raw_ds[0]["user"]
_msgs = [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": _probe}]
try:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
except TypeError:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
_inputs = tokenizer(_txt, return_tensors="pt").to(model.device)
with torch.no_grad():
    _out = model.generate(**_inputs, max_new_tokens=512, do_sample=False, temperature=None, top_p=None)
_gen = tokenizer.decode(_out[0][_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(_gen[:1200])
print("\nHAS_BOXED:", "\\boxed{" in _gen)
try: model.config.use_cache = False
except Exception: pass
model.train()

## Package submission.zip

In [ ]:
import json, shutil, zipfile, os, glob, sys, subprocess

needed = ["adapter_config.json", "adapter_model.safetensors"]
src_dir = SFT_ADAPTER_DIR
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUTPUT_ROOT
print(f"[package] WORKING = {WORKING}")

def _have_all(d):
    return all(os.path.exists(os.path.join(d, n)) for n in needed)

ckpts = []
if not _have_all(src_dir):
    print(f"[package] {needed} not all present in {src_dir}")
    output_dir = sft_config.output_dir if "sft_config" in dir() else None
    ckpts = sorted(
        glob.glob(os.path.join(output_dir, "checkpoint-*")) if output_dir else [],
        key=lambda p: int(p.rsplit("-", 1)[-1]),
    )
    if ckpts:
        ck = ckpts[-1]
        print(f"[package] copying from latest checkpoint: {ck}")
        os.makedirs(src_dir, exist_ok=True)
        for fname in needed:
            sp = os.path.join(ck, fname)
            if os.path.exists(sp):
                shutil.copy2(sp, os.path.join(src_dir, fname))
    if not _have_all(src_dir) and "trainer" in dir():
        print(f"[package] still missing -- attempting trainer.model.save_pretrained({src_dir})")
        try:
            trainer.model.save_pretrained(src_dir)
            tokenizer.save_pretrained(src_dir)
        except Exception as e:
            print(f"[package] live save failed: {e}")

missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
if missing:
    raise FileNotFoundError(
        f"Adapter files {missing} still missing after recovery attempts.\n"
        f"  Checked: {src_dir}\n"
        f"  Available checkpoints: {ckpts if ckpts else 'none'}\n"
    )

POSTPROC_BOOST = float(os.environ.get("POSTPROC_BOOST", "1.0"))
POSTPROC_TOP_FRAC = float(os.environ.get("POSTPROC_TOP_FRAC", "0.5"))
processed_dir = src_dir + "_processed"

script_path = None
for cand in ["tools/postprocess_adapter.py",
             "/kaggle/working/tools/postprocess_adapter.py",
             os.path.join(os.getcwd(), "tools", "postprocess_adapter.py")]:
    if os.path.exists(cand):
        script_path = cand
        break

if script_path:
    cmd = [sys.executable, script_path, "--in", src_dir, "--out", processed_dir,
           "--boost", str(POSTPROC_BOOST), "--top-frac", str(POSTPROC_TOP_FRAC), "--rank", "32"]
    print(f"[package] running adapter post-processing: {' '.join(cmd)}")
    rc = subprocess.run(cmd, check=False).returncode
    if rc != 0 or not all(os.path.exists(os.path.join(processed_dir, n)) for n in needed):
        print(f"[package] post-processing failed (rc={rc}); falling back to raw adapter")
        processed_dir = src_dir
else:
    print(f"[package] tools/postprocess_adapter.py NOT FOUND; using raw adapter")
    processed_dir = src_dir

os.makedirs(SUBMISSION_DIR, exist_ok=True)
for fname in needed:
    sp = os.path.join(processed_dir, fname)
    dp = os.path.join(SUBMISSION_DIR, fname)
    shutil.copy2(sp, dp)
    print(f"  copied {fname}  ({os.path.getsize(dp)/1024/1024:.1f} MB)")

cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

zip_path = os.path.join(WORKING, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in needed:
        zf.write(os.path.join(SUBMISSION_DIR, fname), fname)
print(f"\nsubmission.zip: {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB) - ready "
      f"(boost={POSTPROC_BOOST}  top_frac={POSTPROC_TOP_FRAC}).")